# 15 — Full Agent：把所有模块组装成完整系统

这是最后一章。把前 14 章实现的每个模块按正确顺序装配起来，形成一个完整可运行的 AI Coding Agent。

完成后可以运行：
```bash
python course/main.py
```
然后用中文告诉它任何代码任务，它能自主完成。

---

## 系统全景

```
用户输入（CLI / 交互）
         │
         ▼
┌─────────────────────────────────────────────┐
│              create_agent()                 │
│  LLMClient + ContextManager + ToolRegistry  │
│  ApprovalManager  PersistenceManager        │
│  HookSystem       LoopDetector              │
│  SubAgents        MCPManager（可选）         │
└─────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────┐
│            agentic_loop()                   │
│  每轮：compress_if_needed                   │
│        LLM 调用                             │
│        approval → before_tool hook          │
│        loop_detector.record + check         │
│        tool.invoke                          │
│        after_tool hook                      │
└─────────────────────────────────────────────┘
         │
         ▼
最终回答 + 性能统计
```

In [ ]:
import sys
import os

# 确保能导入 src/
sys.path.insert(0, "..")
os.chdir("..")

# 验证基础模块
try:
    from src.llm_client import LLMClient, TokenUsage
    from src.context_manager import ContextManager, build_system_prompt
    from src.tool_framework import ToolRegistry, ToolResult, Tool, ToolKind
    print("核心模块导入成功")
except ImportError as e:
    print(f"导入失败：{e}")
    print("请确认已运行前面各章节并保存了 src/ 下的模块")

---

## Part 1：完整初始化流程

`create_agent()` 是整个系统的装配工厂。它按固定顺序初始化每个模块，隔离失败（某个模块不存在时打印警告但继续运行），最后返回一个包含所有组件的 `session` 字典。

```
.ai_agent/config.toml
        │
        ▼
  _load_config()     ← 不存在则返回 {}
        │
        ├─ LLMClient(model, base_url, api_key)
        ├─ ContextManager(system_prompt)
        ├─ ToolRegistry
        │     ├─ 文件工具（ReadFile/WriteFile/Edit/ListDir/Glob）
        │     └─ 网络工具（WebSearch/WebFetch）
        ├─ ToolDiscovery（可选）
        ├─ SubAgents（CodebaseInvestigator / CodeReviewer）
        ├─ ApprovalManager(policy)
        ├─ PersistenceManager(store_dir)
        ├─ HookSystem(hooks_config)
        ├─ LoopDetector(max_exact_repeats, max_cycle_length)
        └─ MCPManager（可选）
```

In [ ]:
import asyncio
import json
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional


def _load_config(cwd: Path) -> dict:
    """加载 .ai_agent/config.toml，不存在则返回空字典。"""
    config_path = cwd / ".ai_agent" / "config.toml"
    if not config_path.exists():
        return {}
    try:
        try:
            import tomllib
        except ImportError:
            import tomli as tomllib  # type: ignore
        with open(config_path, "rb") as f:
            return tomllib.load(f)
    except Exception as e:
        print(f"[警告] 配置文件加载失败：{e}")
        return {}


def _register_file_tools(registry, cwd: Path):
    """注册所有文件工具。"""
    try:
        from src.file_tools import (
            ReadFileTool, WriteFileTool, EditTool,
            ListDirectoryTool, GlobTool,
        )
        for tool_cls in [ReadFileTool, WriteFileTool, EditTool,
                         ListDirectoryTool, GlobTool]:
            registry.register(tool_cls(cwd))
        print(f"  [OK] 文件工具注册成功（5 个）")
    except ImportError as e:
        print(f"  [跳过] 文件工具：{e}")


def _register_network_tools(registry):
    """注册网络工具。"""
    try:
        from src.network_tools import WebSearchTool, WebFetchTool
        registry.register(WebSearchTool())
        registry.register(WebFetchTool())
        print("  [OK] 网络工具注册成功（2 个）")
    except ImportError as e:
        print(f"  [跳过] 网络工具：{e}")


async def _run_discovery(registry, cwd: Path) -> list:
    """运行 ToolDiscovery，发现并注册额外工具。"""
    try:
        from src.tool_discovery import ToolDiscovery
        discovery = ToolDiscovery(cwd=cwd)
        tools = await discovery.discover()
        for tool in tools:
            registry.register(tool)
        print(f"  [OK] ToolDiscovery 发现 {len(tools)} 个额外工具")
        return tools
    except (ImportError, Exception) as e:
        print(f"  [跳过] ToolDiscovery：{e}")
        return []


def _build_sub_agents(llm, registry) -> dict:
    """构建子代理。"""
    agents = {}
    try:
        from src.sub_agents import CodebaseInvestigator, CodeReviewer
        agents["investigator"] = CodebaseInvestigator(llm=llm, registry=registry)
        agents["reviewer"] = CodeReviewer(llm=llm, registry=registry)
        print(f"  [OK] 子代理注册成功（investigator, reviewer）")
    except (ImportError, Exception) as e:
        print(f"  [跳过] 子代理：{e}")
    return agents


def _build_approval_manager(policy: str):
    """构建 ApprovalManager。"""
    try:
        from src.approval_and_safety import ApprovalManager
        mgr = ApprovalManager(policy=policy)
        print(f"  [OK] ApprovalManager 初始化成功（策略={policy}）")
        return mgr
    except (ImportError, Exception) as e:
        print(f"  [跳过] ApprovalManager：{e}")
        return None


def _build_persistence(cwd: Path):
    """构建 PersistenceManager。"""
    try:
        from src.persistence_and_checkpoint import PersistenceManager
        store_dir = cwd / ".ai_agent" / "sessions"
        store_dir.mkdir(parents=True, exist_ok=True)
        mgr = PersistenceManager(store_dir=store_dir)
        print(f"  [OK] PersistenceManager 初始化成功（目录={store_dir}）")
        return mgr
    except (ImportError, Exception) as e:
        print(f"  [跳过] PersistenceManager：{e}")
        return None


def _build_hook_system(hooks_config: list):
    """从配置列表构建 HookSystem。"""
    try:
        from src.hooks_and_loop_detection import HookSystem, HookConfig, HookTrigger
        configs = []
        for item in hooks_config:
            try:
                configs.append(HookConfig(
                    trigger=HookTrigger(item["trigger"]),
                    command=item["command"],
                    timeout=float(item.get("timeout", 10.0)),
                    enabled=bool(item.get("enabled", True)),
                ))
            except (KeyError, ValueError) as e:
                print(f"  [警告] 跳过无效 hook 配置：{e}")
        mgr = HookSystem(configs)
        print(f"  [OK] HookSystem 初始化成功（{len(configs)} 个 hook）")
        return mgr
    except (ImportError, Exception) as e:
        print(f"  [跳过] HookSystem：{e}")
        return None


async def _build_mcp_manager(servers_config: dict):
    """构建 MCPManager（可选）。"""
    try:
        from src.mcp_protocol import MCPManager
        mgr = MCPManager(servers=servers_config)
        await mgr.connect_all()
        print(f"  [OK] MCPManager 初始化成功")
        return mgr
    except (ImportError, Exception) as e:
        print(f"  [跳过] MCPManager：{e}")
        return None


async def create_agent(
    model: str = "gpt-oss:120b",
    cwd: Path = Path("."),
    approval_policy: str = "on_request",
    enable_mcp: bool = False,
    enable_discovery: bool = True,
) -> dict:
    """
    完整初始化流程，返回装配好的 session 字典。
    每个模块独立初始化，失败时打印警告但继续装配其他模块。
    """
    print("[create_agent] 开始初始化...")

    # 1. 加载配置
    config = _load_config(cwd)
    config.setdefault("model", model)
    config.setdefault("approval_policy", approval_policy)
    print(f"  [OK] 配置加载完成（{len(config)} 项）")

    # 2. LLMClient + ContextManager
    from src.llm_client import LLMClient
    from src.context_manager import ContextManager, build_system_prompt
    llm = LLMClient(
        model=config.get("model", model),
        base_url=config.get("base_url", "http://localhost:11434/v1"),
        api_key=config.get("api_key", "ollama"),
    )
    system_prompt = build_system_prompt(working_directory=str(cwd))
    ctx = ContextManager(system_prompt=system_prompt)
    print(f"  [OK] LLMClient + ContextManager 初始化成功")

    # 3. ToolRegistry + 工具注册
    from src.tool_framework import ToolRegistry
    registry = ToolRegistry()
    _register_file_tools(registry, cwd)
    _register_network_tools(registry)

    # 4. ToolDiscovery（可选）
    discovered_tools = []
    if enable_discovery:
        discovered_tools = await _run_discovery(registry, cwd)

    # 5. 子代理
    sub_agents = _build_sub_agents(llm, registry)

    # 6. ApprovalManager
    approval_mgr = _build_approval_manager(config.get("approval_policy", approval_policy))

    # 7. PersistenceManager
    persistence = _build_persistence(cwd)

    # 8. HookSystem
    hook_system = _build_hook_system(config.get("hooks", []))

    # 9. LoopDetector
    from src.hooks_and_loop_detection import LoopDetector
    loop_detector = LoopDetector(
        max_exact_repeats=config.get("max_exact_repeats", 3),
        max_cycle_length=config.get("max_cycle_length", 4),
    )
    print(f"  [OK] LoopDetector 初始化成功")

    # 10. MCPManager（可选）
    mcp_manager = None
    if enable_mcp:
        mcp_manager = await _build_mcp_manager(config.get("mcp_servers", {}))

    total_tools = len(registry.list_tools())
    print(f"[create_agent] 初始化完成，已注册 {total_tools} 个工具")

    return {
        "llm": llm,
        "ctx": ctx,
        "registry": registry,
        "approval_mgr": approval_mgr,
        "persistence": persistence,
        "hook_system": hook_system,
        "loop_detector": loop_detector,
        "mcp_manager": mcp_manager,
        "sub_agents": sub_agents,
        "config": config,
        "cwd": cwd,
        "discovered_tools": discovered_tools,
    }


# 测试初始化
async def test_create_agent():
    session = await create_agent(
        model="gpt-oss:120b",
        cwd=Path("."),
        approval_policy="on_request",
        enable_mcp=False,
        enable_discovery=True,
    )
    print(f"\nSession 包含的键：{list(session.keys())}")
    print(f"已注册工具：{[t.name for t in session['registry'].list_tools()]}")
    return session


session = await test_create_agent()

---

## Part 2：完整 Agentic Loop

在第 06 章的基础上，加入所有中间件：

```
每轮循环：
  ┌──────────────────────────────────────────────┐
  │ compress_if_needed(ctx, llm)                 │  ← 第 08 章
  │ llm.chat_completion(messages, tools=schemas) │
  │ 解析响应                                     │
  │   if 工具调用：                               │
  │     approval_mgr.request_approval()          │  ← 第 09 章
  │     hook_system.trigger(BEFORE_TOOL)         │  ← 第 14 章
  │     loop_detector.record() + check()         │  ← 第 14 章
  │     registry.invoke(tool_name, params)       │
  │     hook_system.trigger(AFTER_TOOL)          │  ← 第 14 章
  │   if 最终回答：break                          │
  └──────────────────────────────────────────────┘
```

In [ ]:
async def agentic_loop(
    session: dict,
    user_message: str,
    max_turns: int = 20,
    verbose: bool = True,
) -> dict:
    """
    完整 agentic loop，整合所有中间件。
    返回包含 response / turns / tools_called / elapsed_sec / compressed / token_usage 的字典。
    """
    llm          = session["llm"]
    ctx          = session["ctx"]
    registry     = session["registry"]
    approval_mgr = session.get("approval_mgr")
    hook_system  = session.get("hook_system")
    loop_det     = session.get("loop_detector")

    # 尝试导入压缩函数
    compress_fn = None
    try:
        from src.context_compression import compress_if_needed
        compress_fn = compress_if_needed
    except ImportError:
        pass

    # 尝试导入 HookTrigger
    HookTrigger = None
    try:
        from src.hooks_and_loop_detection import HookTrigger as _HT
        HookTrigger = _HT
    except ImportError:
        pass

    start_time   = time.time()
    tools_called : list[str] = []
    turn         = 0
    compressed   = False
    final_response = ""

    # ── BEFORE_AGENT hook ─────────────────────────────────────────
    if hook_system and HookTrigger:
        await hook_system.trigger(HookTrigger.BEFORE_AGENT)

    ctx.add_user_message(user_message)

    if verbose:
        print(f"\n{'='*60}")
        print(f"[Agent] 任务: {user_message}")
        print(f"[Agent] 可用工具: {[t.name for t in registry.list_tools()]}")
        print(f"{'='*60}")

    for turn in range(max_turns):
        if verbose:
            print(f"\n[轮次 {turn+1}/{max_turns}]")

        # ── 每轮前检查是否需要压缩 ────────────────────────────────
        if compress_fn and ctx.needs_compression():
            if verbose:
                print(f"  [压缩] 触发上下文压缩...")
            try:
                await compress_fn(ctx, llm)
                compressed = True
                if verbose:
                    print(f"  [压缩] 完成")
            except Exception as e:
                if verbose:
                    print(f"  [压缩] 失败（忽略）：{e}")

        # ── 调用 LLM ─────────────────────────────────────────────
        try:
            tools_schema = registry.get_schemas() if registry.list_tools() else None
            response_obj, usage = await llm.chat_completion(
                messages=ctx.get_messages(),
                tools=tools_schema,
            )
            ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)
            if verbose:
                print(f"  [LLM] tokens: prompt={usage.prompt_tokens} completion={usage.completion_tokens}")
        except Exception as e:
            if verbose:
                print(f"  [LLM] 调用失败：{e}")
            if hook_system and HookTrigger:
                await hook_system.trigger(
                    HookTrigger.ON_ERROR,
                    context={"error": str(e)}
                )
            break

        # ── 解析响应：字符串 = 最终回答；dict = 含工具调用 ──────────
        if isinstance(response_obj, str):
            # 纯文本回答，直接结束
            ctx.add_assistant_message(response_obj)
            final_response = response_obj
            if verbose:
                print(f"\n[Agent] 最终回答:")
                print(response_obj)
            break

        if isinstance(response_obj, dict) and "tool_calls" in response_obj:
            content    = response_obj.get("content", "")
            tool_calls = response_obj["tool_calls"]
            ctx.add_assistant_message(content, tool_calls=tool_calls)

            for tc in tool_calls:
                tool_name = tc.get("function", {}).get("name", "")
                raw_args  = tc.get("function", {}).get("arguments", "{}")
                tool_id   = tc.get("id", "")

                try:
                    params = json.loads(raw_args) if isinstance(raw_args, str) else raw_args
                except json.JSONDecodeError:
                    params = {}

                if verbose:
                    params_preview = json.dumps(params, ensure_ascii=False)[:80]
                    print(f"  [工具] {tool_name}({params_preview})")

                # ── ApprovalManager ──────────────────────────────
                if approval_mgr:
                    try:
                        approved = await approval_mgr.request_approval(
                            tool_name=tool_name, params=params
                        )
                        if not approved:
                            ctx.add_tool_result(tool_id, "用户拒绝了此操作")
                            if verbose:
                                print(f"    [审批] 拒绝")
                            continue
                    except Exception:
                        pass

                # ── BEFORE_TOOL hook ─────────────────────────────
                if hook_system and HookTrigger:
                    await hook_system.trigger(
                        HookTrigger.BEFORE_TOOL,
                        context={"tool_name": tool_name}
                    )

                # ── 循环检测 ─────────────────────────────────────
                if loop_det:
                    loop_det.record(tool_name, params)
                    loop_result = loop_det.check()
                    if loop_result:
                        msg = (
                            f"[循环检测] {loop_result.description}\n"
                            f"建议：{loop_result.suggestion}"
                        )
                        if verbose:
                            print(f"    [循环检测] 触发：{loop_result.description}")
                        ctx.add_tool_result(tool_id, msg)
                        continue

                # ── 执行工具 ─────────────────────────────────────
                tool_result = await registry.invoke(tool_name, params)
                tools_called.append(tool_name)
                success = tool_result.success

                if verbose:
                    status  = "OK" if success else "ERR"
                    preview = tool_result.content[:120].replace("\n", " ")
                    print(f"    [{status}] {preview}")

                ctx.add_tool_result(tool_id, tool_result.content)

                # ── AFTER_TOOL hook ──────────────────────────────
                if hook_system and HookTrigger:
                    await hook_system.trigger(
                        HookTrigger.AFTER_TOOL,
                        context={
                            "tool_name": tool_name,
                            "tool_success": str(success).lower(),
                        }
                    )
        else:
            # 未知格式，当作最终回答
            final_response = str(response_obj)
            break

    # ── AFTER_AGENT hook ───────────────────────────────────────────
    if hook_system and HookTrigger:
        await hook_system.trigger(HookTrigger.AFTER_AGENT)

    elapsed = time.time() - start_time
    stats   = ctx.stats()

    return {
        "response":    final_response,
        "turns":       turn + 1,
        "tools_called": tools_called,
        "elapsed_sec": round(elapsed, 2),
        "compressed":  compressed,
        "token_usage": stats,
    }


print("agentic_loop() 定义完成")

---

## Part 3：完整 CLI 命令列表

所有 `/` 命令集中在 `handle_slash_command()` 中处理，与 agentic loop 完全解耦。

In [ ]:
def handle_slash_command(cmd: str, session: dict) -> bool:
    """
    处理 / 命令。
    返回 True 表示命令已处理，False 表示未识别。

    支持命令一览：
      /help                  显示帮助
      /exit                  退出
      /clear                 清空对话历史
      /config                显示当前配置
      /model <name>          切换模型
      /approval <policy>     切换审批策略
      /stats                 显示 token 统计
      /sessions              列出已保存 Session
      /save                  保存当前 Session
      /resume <id>           恢复 Session
      /tools                 列出已注册工具
      /mcp                   列出 MCP 工具
      /checkpoint            创建检查点
      /restore <id>          恢复检查点
    """
    import sys
    parts   = cmd.strip().split(None, 1)
    command = parts[0].lower()
    arg     = parts[1] if len(parts) > 1 else ""

    ctx         = session.get("ctx")
    registry    = session.get("registry")
    persistence = session.get("persistence")
    mcp_manager = session.get("mcp_manager")
    config      = session.get("config", {})

    if command == "/help":
        print("""
可用命令：
  /help                  显示此帮助
  /exit                  退出 Agent
  /clear                 清空对话历史
  /config                显示当前配置
  /model <name>          切换模型（如 /model llama3:8b）
  /approval <policy>     切换审批策略（on_request/auto/autoEdit/never/YOLO）
  /stats                 显示 token 统计
  /sessions              列出已保存的 Session
  /save                  保存当前 Session
  /resume <id>           恢复指定 Session
  /tools                 列出已注册工具
  /mcp                   列出 MCP 工具（如已启用）
  /checkpoint            创建检查点
  /restore <id>          恢复到指定检查点
""")
        return True

    elif command == "/exit":
        print("退出 Agent。")
        sys.exit(0)

    elif command == "/clear":
        if ctx:
            ctx.clear_messages()
            print("对话历史已清空。")
        return True

    elif command == "/config":
        print("当前配置：")
        for k, v in config.items():
            print(f"  {k}: {v}")
        return True

    elif command == "/model":
        if not arg:
            print("用法：/model <model_name>")
        else:
            if session.get("llm"):
                session["llm"].model = arg
                config["model"] = arg
                print(f"模型已切换为：{arg}")
        return True

    elif command == "/approval":
        valid = {"on_request", "auto", "autoEdit", "never", "YOLO"}
        if arg not in valid:
            print(f"无效策略，可选：{', '.join(sorted(valid))}")
        else:
            config["approval_policy"] = arg
            if session.get("approval_mgr"):
                session["approval_mgr"].policy = arg
            print(f"审批策略已切换为：{arg}")
        return True

    elif command == "/stats":
        if ctx:
            stats = ctx.stats()
            print("Token 统计：")
            for k, v in stats.items():
                print(f"  {k}: {v}")
        return True

    elif command == "/sessions":
        if persistence:
            try:
                sessions = persistence.list_sessions()
                if sessions:
                    for s in sessions:
                        print(f"  {s}")
                else:
                    print("无已保存的 Session。")
            except Exception as e:
                print(f"列出 Session 失败：{e}")
        else:
            print("PersistenceManager 未初始化。")
        return True

    elif command == "/save":
        if persistence and ctx:
            try:
                sid = persistence.save_session(ctx)
                print(f"Session 已保存，ID：{sid}")
            except Exception as e:
                print(f"保存失败：{e}")
        else:
            print("PersistenceManager 未初始化。")
        return True

    elif command == "/resume":
        if not arg:
            print("用法：/resume <session_id>")
        elif persistence and ctx:
            try:
                persistence.load_session(arg, ctx)
                print(f"Session {arg} 已恢复。")
            except Exception as e:
                print(f"恢复失败：{e}")
        else:
            print("PersistenceManager 未初始化。")
        return True

    elif command == "/tools":
        if registry:
            tools = registry.list_tools()
            print(f"已注册工具（共 {len(tools)} 个）：")
            for t in tools:
                print(f"  {t.name:20s} [{t.kind.value:8s}]  {t.description[:60]}")
        return True

    elif command == "/mcp":
        if mcp_manager:
            try:
                mcp_tools = mcp_manager.list_tools()
                print(f"MCP 工具（共 {len(mcp_tools)} 个）：")
                for t in mcp_tools:
                    print(f"  {t}")
            except Exception as e:
                print(f"获取 MCP 工具失败：{e}")
        else:
            print("MCP 未启用。")
        return True

    elif command == "/checkpoint":
        if persistence and ctx:
            try:
                cid = persistence.create_checkpoint(ctx)
                print(f"检查点已创建，ID：{cid}")
            except Exception as e:
                print(f"创建检查点失败：{e}")
        else:
            print("PersistenceManager 未初始化。")
        return True

    elif command == "/restore":
        if not arg:
            print("用法：/restore <checkpoint_id>")
        elif persistence and ctx:
            try:
                persistence.restore_checkpoint(arg, ctx)
                print(f"已恢复到检查点 {arg}。")
            except Exception as e:
                print(f"恢复检查点失败：{e}")
        else:
            print("PersistenceManager 未初始化。")
        return True

    return False  # 未识别的命令


# 测试所有命令的注册情况
print("测试 /help 命令：")
handle_slash_command("/help", session)

print("\n测试 /tools 命令：")
handle_slash_command("/tools", session)

print("\n测试 /stats 命令：")
handle_slash_command("/stats", session)

print("\n测试 /config 命令：")
handle_slash_command("/config", session)

print("\n测试未知命令：")
result = handle_slash_command("/unknown", session)
print(f"  未识别命令返回: {result}")

---

## Part 4：主入口 main.py

`main.py` 已写好，位于 `course/main.py`，结构如下：

```
course/main.py
  @click.command()           ← CLI 参数解析
  main()                     ← 同步入口，调用 asyncio.run(run(...))
  run()                      ← 顶层协程：create_agent + 交互/单次执行
  interactive_loop()         ← REPL 主循环
  handle_slash_command()     ← / 命令处理器
  agentic_loop()             ← 完整 agentic loop
  create_agent()             ← 装配工厂
```

运行方式：
```bash
# 交互模式
python course/main.py

# 单次执行
python course/main.py -p "列出 src/ 目录的所有 Python 文件"

# 切换模型
python course/main.py -m llama3:8b

# 关闭自动审批（每次工具调用都需要确认）
python course/main.py --approval on_request

# 完全信任模式
python course/main.py --approval YOLO
```

下面验证 main.py 可以正确加载（不运行）：

In [ ]:
import importlib.util
import sys

main_path = Path("course/main.py")

if main_path.exists():
    spec = importlib.util.spec_from_file_location("course.main", main_path)
    module = importlib.util.module_from_spec(spec)
    # 不执行，只加载检查语法
    try:
        spec.loader.exec_module(module)
        print(f"[OK] course/main.py 语法检查通过")
        # 检查关键函数是否存在
        for name in ["main", "run", "agentic_loop", "create_agent", "handle_slash_command"]:
            if hasattr(module, name):
                print(f"  [OK] {name}() 存在")
            else:
                print(f"  [缺失] {name}()")
    except SystemExit:
        print(f"[OK] course/main.py 加载成功（Click 正常执行了入口保护）")
    except Exception as e:
        print(f"[ERROR] {e}")
else:
    print(f"[ERROR] course/main.py 不存在")

---

## Part 5：集成测试——真实端到端任务

给 Agent 一个完整任务，真实调用 Ollama，观察整个执行过程。

**任务**：分析 `src/` 目录里的代码，找出所有类的名称和它们的用途，生成一个 `SUMMARY.md` 文件。

这个任务需要：
1. `list_directory` 看目录结构
2. `glob` 找所有 Python 文件
3. `read_file` 逐个读取
4. `write_file` 生成 SUMMARY.md

> **注意**：此 cell 真实调用 Ollama。运行前确保 Ollama 已启动：`ollama serve`
> 如果 Ollama 未启动，cell 会打印连接错误并退出，不会崩溃。

In [ ]:
async def run_integration_test():
    """
    端到端集成测试：真实调用 Ollama，完成代码分析任务。
    打印每个事件，最后输出性能统计。
    """
    print("="*60)
    print("集成测试：代码分析任务")
    print("="*60)

    # 为测试创建独立 session（避免污染前面的全局 session）
    test_session = await create_agent(
        model="gpt-oss:120b",
        cwd=Path("."),
        approval_policy="YOLO",   # 测试中自动批准所有工具调用
        enable_mcp=False,
        enable_discovery=False,   # 跳过 Discovery，加快启动
    )

    task = (
        "请分析 src/ 目录里的 Python 代码，"
        "找出所有 class 的名称和它们的用途，"
        "然后生成一个 SUMMARY.md 文件，内容包括："
        "1) 每个类的名称；2) 所在文件；3) 一句话描述用途。"
        "使用 Markdown 表格格式。"
    )

    result = await agentic_loop(
        session=test_session,
        user_message=task,
        max_turns=25,
        verbose=True,
    )

    # 关闭连接
    await test_session["llm"].close()

    return result


# 尝试运行，捕获连接错误
try:
    test_result = await run_integration_test()
except Exception as e:
    print(f"\n[测试终止] {type(e).__name__}: {e}")
    print("请确认 Ollama 已启动：ollama serve")
    test_result = None

---

## Part 6：性能数据

打印任务完成后的完整性能摘要。

In [ ]:
def print_performance_summary(result: dict | None):
    """
    打印完整的性能摘要。
    result 是 agentic_loop() 的返回值。
    """
    if result is None:
        print("无结果（任务未执行或已中断）")
        return

    print("\n" + "="*60)
    print("性能摘要")
    print("="*60)

    # 总轮次数
    print(f"  总轮次数:       {result['turns']}")

    # 耗时
    print(f"  总耗时:         {result['elapsed_sec']} 秒")

    # 工具调用统计
    tools = result["tools_called"]
    if tools:
        from collections import Counter
        tool_counts = Counter(tools)
        print(f"  工具调用总数:   {len(tools)} 次")
        print(f"  工具调用列表:")
        for tool_name, count in sorted(tool_counts.items(), key=lambda x: -x[1]):
            print(f"    {tool_name:20s} x{count}")
    else:
        print(f"  工具调用:       无")

    # Token 消耗
    usage = result.get("token_usage", {})
    if usage:
        print(f"  Token 统计:")
        for k, v in usage.items():
            print(f"    {k}: {v}")

    # 是否触发压缩
    print(f"  触发上下文压缩: {'是' if result['compressed'] else '否'}")

    # 最终回答（截取前 200 字符）
    response = result.get("response", "")
    if response:
        preview = response[:200].replace("\n", " ")
        suffix  = "..." if len(response) > 200 else ""
        print(f"  最终回答预览:   {preview}{suffix}")

    print("="*60)


# 打印测试结果
print_performance_summary(test_result)

# 如果 SUMMARY.md 生成了，显示其内容
summary_path = Path("SUMMARY.md")
if summary_path.exists():
    print(f"\nSUMMARY.md 内容：")
    print("-"*60)
    print(summary_path.read_text(encoding="utf-8")[:800])
    print("-"*60)
else:
    print("\nSUMMARY.md 未生成（任务可能未完成或 Ollama 未启动）")

---

## Part 7：错误场景处理

生产环境中最常见的三种失败模式，以及系统如何应对。

In [ ]:
# ── 场景 1：LLM 连接失败 ──────────────────────────────────────────
async def test_llm_connection_failure():
    """
    测试：LLM 地址无效时，agentic_loop 应该捕获错误并返回，
    而不是崩溃整个程序。
    """
    print("场景 1：LLM 连接失败")
    print("-"*40)

    from src.llm_client import LLMClient
    from src.context_manager import ContextManager, build_system_prompt
    from src.tool_framework import ToolRegistry

    # 故意使用无效端口
    bad_llm = LLMClient(
        model="gpt-oss:120b",
        base_url="http://localhost:19999/v1",  # 不存在的端口
        api_key="ollama",
    )
    ctx = ContextManager(system_prompt=build_system_prompt())
    registry = ToolRegistry()

    fake_session = {
        "llm": bad_llm,
        "ctx": ctx,
        "registry": registry,
        "approval_mgr": None,
        "hook_system": None,
        "loop_detector": None,
    }

    result = await agentic_loop(
        session=fake_session,
        user_message="列出当前目录",
        max_turns=3,
        verbose=False,
    )
    await bad_llm.close()

    print(f"  连接失败后 agentic_loop 正常返回: True")
    print(f"  轮次数: {result['turns']}")
    print(f"  最终回答: {result['response']!r}")


# ── 场景 2：工具不存在 ────────────────────────────────────────────
async def test_missing_tool():
    """
    测试：调用不存在的工具时，ToolRegistry 应该返回 ToolResult.error，
    而不是抛出异常。
    """
    print("\n场景 2：调用不存在的工具")
    print("-"*40)

    from src.tool_framework import ToolRegistry
    registry = ToolRegistry()

    result = await registry.invoke("nonexistent_tool", {"arg": "val"})
    print(f"  success: {result.success}")
    print(f"  is_error: {result.is_error}")
    print(f"  content: {result.content!r}")


# ── 场景 3：循环检测触发 ──────────────────────────────────────────
def test_loop_detection_inline():
    """
    测试：在 agentic_loop 中，循环检测如何把错误消息注入对话。
    """
    print("\n场景 3：循环检测注入错误消息")
    print("-"*40)

    from src.hooks_and_loop_detection import LoopDetector
    detector = LoopDetector(max_exact_repeats=3)

    # 模拟连续三次相同工具调用
    for i in range(3):
        detector.record("read_file", {"path": "/tmp/a.py"})

    loop_result = detector.check()
    if loop_result:
        injected_msg = (
            f"[循环检测] {loop_result.description}\n"
            f"建议：{loop_result.suggestion}"
        )
        print(f"  循环检测触发: True")
        print(f"  注入消息预览: {injected_msg[:100]}")
    else:
        print("  循环检测未触发（不符合预期）")


await test_llm_connection_failure()
await test_missing_tool()
test_loop_detection_inline()

---

## 小结

| 章节 | 模块 | 核心价值 |
|------|------|----------|
| 01 | `LLMClient` | 封装 OpenAI 兼容 API，支持流式、重试、token 统计 |
| 02 | `ContextManager` | 管理对话历史，追踪 token，判断是否需要压缩 |
| 03 | `ToolFramework` | 定义 Tool/ToolRegistry/ToolResult 统一接口 |
| 04 | 文件工具 | ReadFile/WriteFile/Edit/ListDir/Glob，安全沙箱路径 |
| 05 | 网络工具 | WebSearch/WebFetch，带超时和内容截断 |
| 06 | `agentic_loop` | 基础工具调用循环，解析 LLM 的 function_call 响应 |
| 07 | Session + CLI | 会话对象 + `/help` 等基础命令 |
| 08 | 上下文压缩 | 超出阈值时自动摘要旧消息，保持 context window 可用 |
| 09 | `ApprovalManager` | 五种审批策略，写操作需要用户确认 |
| 10 | `PersistenceManager` | Session 存档/恢复，检查点机制 |
| 11 | 子代理 | CodebaseInvestigator / CodeReviewer，专用子任务 |
| 12 | MCP 协议 | 连接外部 MCP 服务器，动态扩展工具集 |
| 13 | `ToolDiscovery` | 自动发现本地可执行文件并封装为工具 |
| 14 | HookSystem + LoopDetector | 生命周期钩子 + 循环模式检测，防止失控 |
| 15 | Full Agent | 把所有模块装配为完整系统，提供 CLI 入口 |

从这里开始可以做什么：加新工具（实现 `Tool` 基类，一行 `register()` 即可）；接 MCP（配置 `mcp_servers`，无需改核心代码）；改系统 prompt（修改 `build_system_prompt()`）；部署为服务（在 `run()` 前加 FastAPI 路由）；接入任何 OpenAI 兼容模型（换 `base_url` 和 `model` 即可）。

In [ ]:
# 将本章核心代码保存到 src/full_agent.py
import os

os.makedirs("src", exist_ok=True)

code = '''"""
src/full_agent.py — Full Agent 装配模块

对外暴露两个主要函数：
  - create_agent()   完整初始化，返回 session 字典
  - agentic_loop()   完整工具调用循环，集成所有中间件
"""
import asyncio
import json
import time
from pathlib import Path
from typing import Optional


def _load_config(cwd: Path) -> dict:
    config_path = cwd / ".ai_agent" / "config.toml"
    if not config_path.exists():
        return {}
    try:
        try:
            import tomllib
        except ImportError:
            import tomli as tomllib
        with open(config_path, "rb") as f:
            return tomllib.load(f)
    except Exception as e:
        print(f"[警告] 配置文件加载失败：{e}")
        return {}


def _register_file_tools(registry, cwd: Path):
    try:
        from src.file_tools import (
            ReadFileTool, WriteFileTool, EditTool,
            ListDirectoryTool, GlobTool,
        )
        for cls in [ReadFileTool, WriteFileTool, EditTool, ListDirectoryTool, GlobTool]:
            registry.register(cls(cwd))
    except ImportError as e:
        print(f"[跳过] 文件工具：{e}")


def _register_network_tools(registry):
    try:
        from src.network_tools import WebSearchTool, WebFetchTool
        registry.register(WebSearchTool())
        registry.register(WebFetchTool())
    except ImportError as e:
        print(f"[跳过] 网络工具：{e}")


async def _run_discovery(registry, cwd: Path) -> list:
    try:
        from src.tool_discovery import ToolDiscovery
        tools = await ToolDiscovery(cwd=cwd).discover()
        for t in tools:
            registry.register(t)
        return tools
    except Exception as e:
        print(f"[跳过] ToolDiscovery：{e}")
        return []


def _build_sub_agents(llm, registry) -> dict:
    try:
        from src.sub_agents import CodebaseInvestigator, CodeReviewer
        return {
            "investigator": CodebaseInvestigator(llm=llm, registry=registry),
            "reviewer":     CodeReviewer(llm=llm, registry=registry),
        }
    except Exception:
        return {}


def _build_approval_manager(policy: str):
    try:
        from src.approval_and_safety import ApprovalManager
        return ApprovalManager(policy=policy)
    except Exception:
        return None


def _build_persistence(cwd: Path):
    try:
        from src.persistence_and_checkpoint import PersistenceManager
        store_dir = cwd / ".ai_agent" / "sessions"
        store_dir.mkdir(parents=True, exist_ok=True)
        return PersistenceManager(store_dir=store_dir)
    except Exception:
        return None


def _build_hook_system(hooks_config: list):
    try:
        from src.hooks_and_loop_detection import HookSystem, HookConfig, HookTrigger
        configs = []
        for item in hooks_config:
            try:
                configs.append(HookConfig(
                    trigger=HookTrigger(item["trigger"]),
                    command=item["command"],
                    timeout=float(item.get("timeout", 10.0)),
                    enabled=bool(item.get("enabled", True)),
                ))
            except (KeyError, ValueError):
                pass
        return HookSystem(configs)
    except Exception:
        return None


async def _build_mcp_manager(servers_config: dict):
    try:
        from src.mcp_protocol import MCPManager
        mgr = MCPManager(servers=servers_config)
        await mgr.connect_all()
        return mgr
    except Exception:
        return None


async def create_agent(
    model: str = "gpt-oss:120b",
    cwd: Path = Path("."),
    approval_policy: str = "on_request",
    enable_mcp: bool = False,
    enable_discovery: bool = True,
) -> dict:
    """完整初始化流程，返回装配好的 session 字典。"""
    config = _load_config(cwd)
    config.setdefault("model", model)
    config.setdefault("approval_policy", approval_policy)

    from src.llm_client import LLMClient
    from src.context_manager import ContextManager, build_system_prompt
    from src.tool_framework import ToolRegistry

    llm = LLMClient(
        model=config.get("model", model),
        base_url=config.get("base_url", "http://localhost:11434/v1"),
        api_key=config.get("api_key", "ollama"),
    )
    ctx = ContextManager(system_prompt=build_system_prompt(working_directory=str(cwd)))
    registry = ToolRegistry()

    _register_file_tools(registry, cwd)
    _register_network_tools(registry)

    discovered_tools = []
    if enable_discovery:
        discovered_tools = await _run_discovery(registry, cwd)

    from src.hooks_and_loop_detection import LoopDetector

    return {
        "llm":             llm,
        "ctx":             ctx,
        "registry":        registry,
        "approval_mgr":    _build_approval_manager(config.get("approval_policy", approval_policy)),
        "persistence":     _build_persistence(cwd),
        "hook_system":     _build_hook_system(config.get("hooks", [])),
        "loop_detector":   LoopDetector(
                               max_exact_repeats=config.get("max_exact_repeats", 3),
                               max_cycle_length=config.get("max_cycle_length", 4),
                           ),
        "mcp_manager":     await _build_mcp_manager(config.get("mcp_servers", {})) if enable_mcp else None,
        "sub_agents":      _build_sub_agents(llm, registry),
        "config":          config,
        "cwd":             cwd,
        "discovered_tools": discovered_tools,
    }


async def agentic_loop(
    session: dict,
    user_message: str,
    max_turns: int = 20,
    verbose: bool = True,
) -> dict:
    """完整 agentic loop，集成压缩、审批、hooks、循环检测。"""
    llm          = session["llm"]
    ctx          = session["ctx"]
    registry     = session["registry"]
    approval_mgr = session.get("approval_mgr")
    hook_system  = session.get("hook_system")
    loop_det     = session.get("loop_detector")

    compress_fn = None
    try:
        from src.context_compression import compress_if_needed
        compress_fn = compress_if_needed
    except ImportError:
        pass

    HookTrigger = None
    try:
        from src.hooks_and_loop_detection import HookTrigger as _HT
        HookTrigger = _HT
    except ImportError:
        pass

    start_time     = time.time()
    tools_called   : list[str] = []
    turn           = 0
    compressed     = False
    final_response = ""

    if hook_system and HookTrigger:
        await hook_system.trigger(HookTrigger.BEFORE_AGENT)

    ctx.add_user_message(user_message)

    for turn in range(max_turns):
        if compress_fn and ctx.needs_compression():
            try:
                await compress_fn(ctx, llm)
                compressed = True
            except Exception:
                pass

        try:
            tools_schema = registry.get_schemas() if registry.list_tools() else None
            response_obj, usage = await llm.chat_completion(
                messages=ctx.get_messages(),
                tools=tools_schema,
            )
            ctx.update_usage(usage.prompt_tokens, usage.completion_tokens)
        except Exception as e:
            if verbose:
                print(f"[LLM 错误] {e}")
            if hook_system and HookTrigger:
                await hook_system.trigger(HookTrigger.ON_ERROR, context={"error": str(e)})
            break

        if isinstance(response_obj, str):
            ctx.add_assistant_message(response_obj)
            final_response = response_obj
            break

        if isinstance(response_obj, dict) and "tool_calls" in response_obj:
            ctx.add_assistant_message(
                response_obj.get("content", ""),
                tool_calls=response_obj["tool_calls"]
            )
            for tc in response_obj["tool_calls"]:
                tool_name = tc.get("function", {}).get("name", "")
                raw_args  = tc.get("function", {}).get("arguments", "{}")
                tool_id   = tc.get("id", "")
                try:
                    params = json.loads(raw_args) if isinstance(raw_args, str) else raw_args
                except json.JSONDecodeError:
                    params = {}

                if approval_mgr:
                    try:
                        if not await approval_mgr.request_approval(tool_name=tool_name, params=params):
                            ctx.add_tool_result(tool_id, "用户拒绝了此操作")
                            continue
                    except Exception:
                        pass

                if hook_system and HookTrigger:
                    await hook_system.trigger(HookTrigger.BEFORE_TOOL, context={"tool_name": tool_name})

                if loop_det:
                    loop_det.record(tool_name, params)
                    loop_result = loop_det.check()
                    if loop_result:
                        ctx.add_tool_result(
                            tool_id,
                            f"[循环检测] {loop_result.description}\n建议：{loop_result.suggestion}"
                        )
                        continue

                tool_result = await registry.invoke(tool_name, params)
                tools_called.append(tool_name)
                ctx.add_tool_result(tool_id, tool_result.content)

                if hook_system and HookTrigger:
                    await hook_system.trigger(
                        HookTrigger.AFTER_TOOL,
                        context={"tool_name": tool_name, "tool_success": str(tool_result.success).lower()}
                    )
        else:
            final_response = str(response_obj)
            break

    if hook_system and HookTrigger:
        await hook_system.trigger(HookTrigger.AFTER_AGENT)

    return {
        "response":     final_response,
        "turns":        turn + 1,
        "tools_called": tools_called,
        "elapsed_sec":  round(time.time() - start_time, 2),
        "compressed":   compressed,
        "token_usage":  ctx.stats(),
    }
'''

with open("src/full_agent.py", "w", encoding="utf-8") as f:
    f.write(code.strip())

print("已保存到 src/full_agent.py")

# 验证导入
try:
    from src.full_agent import create_agent as _ca, agentic_loop as _al
    print("src/full_agent.py 导入验证通过")
except Exception as e:
    print(f"导入验证失败：{e}")